In [1]:
import numpy as np
import pandas 
import dpluspy

In [3]:
"""
Simulated BLRT
"""
models = [
    "hn_null_model",
    "hn_model_1",
    "hn_model_5",
    "sd_null_model",
    "sd_model_1",
    "sd_model_5"
]

focal_df = pandas.read_csv("simulations/bootstrap_lrt_demo/results/data_set_results.csv")
rep_dfs = {}
for model in models:
    for rep in range(10):
        rep_dfs[(model, rep)] = pandas.read_csv(
            f"simulations/bootstrap_lrt_demo/results/{model}_rep_{rep}_samples.csv")


for f, hn_model, sd_model in zip([0, 0.01, 0.05], models[:3], models[3:]):
    for rep in range(10):
        foc_hn = next(iter(focal_df[focal_df["trial"] == f"{hn_model}_rep_{rep}"]["score"]))
        foc_sd = next(iter(focal_df[focal_df["trial"] == f"{sd_model}_rep_{rep}"]["score"]))

        nulldist_hn = np.array(rep_dfs[(hn_model, rep)]["score"])
        nulldist_hn = nulldist_hn[nulldist_hn > 0]
        nulldist_sd = np.array(rep_dfs[(sd_model, rep)]["score"])
        nulldist_sd = nulldist_sd[nulldist_sd > 0]

        p_hn = np.count_nonzero(nulldist_hn > foc_hn) / 100
        p_sd = np.count_nonzero(nulldist_sd > foc_sd) / 100
        print(
            f"& {rep} & {int(np.round(foc_hn, 0))} & {np.round(p_hn, 3)} &" + 
            f"& {rep} & {int(np.round(foc_sd, 0))} & {np.round(p_sd, 3)} \\\\"
        )
   

& 0 & 4 & 0.11 && 0 & 11 & 0.11 \\
& 1 & 1 & 0.3 && 1 & 11 & 0.1 \\
& 2 & 0 & 0.77 && 2 & 14 & 0.07 \\
& 3 & 0 & 0.57 && 3 & 1 & 0.32 \\
& 4 & 0 & 0.36 && 4 & 0 & 0.57 \\
& 5 & 0 & 0.44 && 5 & 0 & 0.46 \\
& 6 & 3 & 0.14 && 6 & 0 & 0.52 \\
& 7 & 0 & 0.53 && 7 & 2 & 0.26 \\
& 8 & 0 & 0.44 && 8 & 0 & 0.4 \\
& 9 & 0 & 0.32 && 9 & 3 & 0.21 \\
& 0 & 0 & 0.54 && 0 & 29 & 0.01 \\
& 1 & 0 & 0.76 && 1 & 18 & 0.06 \\
& 2 & 2 & 0.21 && 2 & 7 & 0.16 \\
& 3 & 0 & 0.8 && 3 & 0 & 0.59 \\
& 4 & 20 & 0.0 && 4 & 150 & 0.0 \\
& 5 & 16 & 0.02 && 5 & 15 & 0.01 \\
& 6 & 13 & 0.05 && 6 & 33 & 0.0 \\
& 7 & 4 & 0.18 && 7 & 7 & 0.18 \\
& 8 & 10 & 0.06 && 8 & 5 & 0.05 \\
& 9 & 18 & 0.03 && 9 & 7 & 0.08 \\
& 0 & 220 & 0.0 && 0 & 149 & 0.0 \\
& 1 & 168 & 0.0 && 1 & 168 & 0.0 \\
& 2 & 117 & 0.0 && 2 & 419 & 0.0 \\
& 3 & 324 & 0.0 && 3 & 181 & 0.0 \\
& 4 & 193 & 0.0 && 4 & 288 & 0.0 \\
& 5 & 16 & 0.08 && 5 & 162 & 0.0 \\
& 6 & 360 & 0.0 && 6 & 239 & 0.0 \\
& 7 & 248 & 0.0 && 7 & 124 & 0.0 \\
& 8 & 134 & 0.0 && 8 & 20

In [7]:
focal_df

,trial,ll0,ll1,score
0,hn_null_model_rep_0,-16.685546,-14.891469,3.588153
1,hn_null_model_rep_1,-16.429874,-15.970587,0.918574
2,hn_null_model_rep_2,-12.556904,-12.575674,0.000000
3,hn_null_model_rep_3,-12.704778,-12.689006,0.031543
4,hn_null_model_rep_4,-14.389864,-14.193316,0.393097
5,hn_null_model_rep_5,-20.318059,-20.261240,0.113638
6,hn_null_model_rep_6,-33.434694,-32.170919,2.527550
7,hn_null_model_rep_7,-28.184679,-28.154927,0.059505
8,hn_null_model_rep_8,-16.393504,-16.352244,0.082519
9,hn_null_model_rep_9,-20.409696,-20.212163,0.395066


25

In [2]:
def printout(mapping, fixed_params, groups):
    for pname in mapping:
        fancyname, fac = mapping[pname]
        out = [fancyname]
        for group in groups:
            whole, boot = group
            mle = np.array(whole[whole["ll"] == np.max(whole["ll"])])[0, 2:].astype(np.float64)
            if pname in whole.columns:
                idx = list(whole.columns[2:]).index(pname)
                pval = mle[idx] * fac
                pstr = f"{np.format_float_positional(pval, precision=3, fractional=False, trim='-')}"
                lower = np.quantile(boot[pname], 0.025) * fac
                upper = np.quantile(boot[pname], 0.975) * fac
                ci_str = (f"{np.format_float_positional(lower, precision=3, fractional=False, trim='-')} "
                        f"-- {np.format_float_positional(upper, precision=3, fractional=False, trim='-')}")
            elif pname in fixed_params:
                pval = fixed_params[pname] * fac
                pstr = f"{np.format_float_positional(pval, precision=3, fractional=False, trim='-')}"
                ci_str = "---"
            else:
                pstr = ""
                ci_str = ""
            out += [pstr, ci_str]
        out = " & ".join(out) + r" \\"
        print(out)
    return

In [3]:
"""
make a table of parameter MLEs and CIs for human/neanderthal introgression models
"""


null_model_mle =dpluspy.inference.load_param_table("models/main_models/params_amh_nea_no_pulse.yaml",
    ["models/main_models/bherer_amh_nea_no_pulse.yaml"])
introgression_model_mle = dpluspy.inference.load_param_table("models/main_models/params_amh_nea_two_pulse.yaml",
    ["models/main_models/bherer_amh_nea_two_pulse.yaml"])
null_model_bootstrap = pandas.read_csv("models/bootstrap_models/bherer_amh_nea_no_pulse.csv")
introgession_model_bootstrap = pandas.read_csv("models/bootstrap_models/bherer_amh_nea_two_pulse.csv")

groups = [
    [null_model_mle, null_model_bootstrap],
    [introgression_model_mle, introgession_model_bootstrap]
]

mapping = {
    "T_NDH": (r"$T_{\text{ND--H}}$ (ka)", 1e-3),
    "T_ND": (r"$T_{\text{N--Den}}$ (ka)", 1e-3),
    "T_AN": (r"$T_{\text{WN--Alt}}$ (ka)", 1e-3),
    "T_CV": (r"$T_{\text{Cha--Vin}}$ (ka)", 1e-3),
    "T_LY": (r"$T_{\text{Yor--Los}}$ (ka)", 1e-3),
    "T_V_L": (r"$T_{\text{Vin}\to\text{Los}}$", 1e-3),
    "N_A": (r"$N_{\text{A}}$", 1),
    "N_ND": (r"$N_{\text{ND}}$ ", 1),
    "N_Den": (r"$N_{\text{Den}}$ ", 1),
    "N_N": (r"$N_{\text{N}}$ ", 1),
    "N_Alt": (r"$N_{\text{Alt}}$", 1),
    "N_Cha": (r"$N_{\text{Cha}}$", 1),
    "N_Vin": (r"$N_{\text{Vin}}$", 1),
    "N_MH": (r"$N_{\text{H}}$ ", 1),
    "N_Yor": (r"$N_{\text{Yor}}$ ", 1),
    "N_Los": (r"$N_{\text{Los}}$ ", 1),
    "m_AD": (r"$m_{\text{N--Den}}$ ($\times 10^{-5}$)", 1e5),
    "m_YL": (r"$m_{\text{Yor--Los}}$ ($\times 10^{-5}$)", 1e5),
    "p_V_L": (r"$\gamma_{\text{Vin} \to \text{Los}}$", 1),

    "T_H_N": (r"$T_{\text{H} \to \text{N}}$ (ka)", 1e-3),
    "T_H_WN": (r"$T_{\text{H} \to \text{WN}}$ (ka)", 1e-3),
    "p_H_N": (r"$\gamma_{\text{H} \to \text{N}}$", 1),
    "p_H_WN": (r"$\gamma_{\text{H} \to \text{WN}}$", 1)
}

fixed_params = {
    "T_H_N": 2.5e5, 
    "T_H_WN": 1.1e5, 
    "T_V_L": 48000, 
}

printout(mapping, fixed_params, groups)

$T_{\text{ND--H}}$ (ka) & 819 & 760 -- 865 & 779 & 731 -- 831 \\
$T_{\text{N--Den}}$ (ka) & 662 & 585 -- 781 & 726 & 654 -- 788 \\
$T_{\text{WN--Alt}}$ (ka) & 123 & 118 -- 133 & 119 & 116 -- 125 \\
$T_{\text{Cha--Vin}}$ (ka) & 62.9 & 58.3 -- 76.1 & 60.5 & 58 -- 66.7 \\
$T_{\text{Yor--Los}}$ (ka) & 48 & 48 -- 48 & 48.1 & 48 -- 66.8 \\
$T_{\text{Vin}\to\text{Los}}$ & 48 & --- & 48 & --- \\
$N_{\text{A}}$ & 16600 & 15900 -- 17500 & 17100 & 16300 -- 17900 \\
$N_{\text{ND}}$  & 12200 & 3630 -- 18400 & 2180 & 807 -- 4850 \\
$N_{\text{Den}}$  & 2830 & 2570 -- 3100 & 3620 & 3150 -- 4000 \\
$N_{\text{N}}$  & 3220 & 2950 -- 3430 & 2610 & 2300 -- 2830 \\
$N_{\text{Alt}}$ & 303 & 127 -- 736 & 189 & 61.7 -- 670 \\
$N_{\text{Cha}}$ & 387 & 140 -- 1110 & 234 & 110 -- 563 \\
$N_{\text{Vin}}$ & 1070 & 729 -- 1870 & 834 & 635 -- 1250 \\
$N_{\text{H}}$  & 52100 & 41500 -- 65200 & 29100 & 26600 -- 32200 \\
$N_{\text{Yor}}$  & 3380 & 2800 -- 4490 & 16900 & 10200 -- 52400 \\
$N_{\text{Los}}$  & 459 & 392 --

In [4]:
"""
for superarchaic introgression models
"""


null_model_mle =dpluspy.inference.load_param_table("models/main_models/params_sup_den_no_pulse.yaml",
    ["models/main_models/bherer_sup_den_no_pulse.yaml"])
introgression_model_mle = dpluspy.inference.load_param_table("models/main_models/params_sup_den.yaml",
    ["models/main_models/bherer_sup_den.yaml"])
null_model_bootstrap = pandas.read_csv("models/bootstrap_models/bherer_sup_den_no_pulse.csv")
introgession_model_bootstrap = pandas.read_csv("models/bootstrap_models/bherer_sup_den.csv")

groups = [
    [null_model_mle, null_model_bootstrap],
    [introgression_model_mle, introgession_model_bootstrap]
]

mapping = {
    'T_NDH': (r'$T_{\text{ND--H}}$ (ka)', 1e-3),
    'T_ND': (r'$T_{\text{N--Den}}$ (ka)', 1e-3),
    "T_HN0": (r"$T_{\text{H} \to \text{N}}$ (ka) (fixed)", 1e-3), 
    'T_AWN': (r'$T_{\text{WN--Alt}}$ (ka)', 1e-3),
    "T_HN1": (r"$T_{\text{H} \to \text{WN}}$ (ka) (fixed)", 1e-3),
    'T_CV': (r'$T_{\text{Cha--Vin}}$ (ka) ', 1e-3),
    'N_Anc': (r'$N_{\text{A}}$', 1),
    'N_ND': (r'$N_{\text{ND}}$', 1),
    'N_N': (r'$N_{\text{N}}$', 1),
    'N_Alt': (r'$N_{\text{Alt}}$', 1),
    'N_Chag': (r'$N_{\text{Cha}}$', 1),
    'N_Vin': (r'$N_{\text{Vin}}$', 1),
    'N_Den': (r'$N_{\text{Den}}$', 1),
    'N_H': (r'$N_{\text{Yor}}$', 1),
    'm_ND': (r'$m_{\text{N--Den}}$ ($\times 10^{-5}$)', 1e5),
    'p_H_N': (r'$f_{\text{H} \to \text{N}}$', 1),
    'p_H_WN': (r'$f_{\text{H} \to \text{WN}}$', 1),
    "T_SA": (r"$T_{\text{NDH--S}}$ (ka) (fixed)", 1e-3),
    "N_S": (r"$N_{\text{S}}$ (fixed)", 1),
    'p_S_D': (r'$f_{\text{S} \to \text{D}}$', 1),
}

fixed_params = {
    "T_S": 2e6,
    "T_HN0": 2.5e5, 
    "T_HN1": 1.1e5, 
    "T_NH": 48000, 
    "T_Ust": 47000, 
    "N_S": 20000, 
}

printout(mapping, fixed_params, groups)

$T_{\text{ND--H}}$ (ka) & 766 & 716 -- 815 & 763 & 709 -- 817 \\
$T_{\text{N--Den}}$ (ka) & 726 & 619 -- 777 & 757 & 607 -- 771 \\
$T_{\text{H} \to \text{N}}$ (ka) (fixed) & 250 & --- & 250 & --- \\
$T_{\text{WN--Alt}}$ (ka) & 119 & 117 -- 164 & 121 & 117 -- 139 \\
$T_{\text{H} \to \text{WN}}$ (ka) (fixed) & 110 & --- & 110 & --- \\
$T_{\text{Cha--Vin}}$ (ka)  & 61 & 58.5 -- 105 & 60.5 & 58.6 -- 67.9 \\
$N_{\text{A}}$ & 17300 & 16400 -- 18100 & 17100 & 16200 -- 17900 \\
$N_{\text{ND}}$ & 1690 & 463 -- 5970 & 311 & 511 -- 7170 \\
$N_{\text{N}}$ & 2590 & 1800 -- 2850 & 2760 & 2520 -- 3060 \\
$N_{\text{Alt}}$ & 205 & 61.6 -- 3020 & 332 & 101 -- 1700 \\
$N_{\text{Cha}}$ & 254 & 138 -- 2240 & 231 & 132 -- 651 \\
$N_{\text{Vin}}$ & 851 & 665 -- 3080 & 819 & 666 -- 1290 \\
$N_{\text{Den}}$ & 3570 & 3070 -- 4090 & 3290 & 2860 -- 3930 \\
$N_{\text{Yor}}$ & 27800 & 26200 -- 29400 & 28500 & 26700 -- 30300 \\
$m_{\text{N--Den}}$ ($\times 10^{-5}$) & 0.27 & 0.001 -- 0.537 & 0.465 & 0.001 -- 0.657 \

In [5]:
"""
print parameter table for amh interaction models
"""


admixture_model_mle =dpluspy.inference.load_param_table("models/main_models/params_los_stu_admixture.yaml",
    ["models/main_models/bherer_los_stu_admixture.yaml"])
migration_model_mle = dpluspy.inference.load_param_table("models/main_models/params_los_stu_migration.yaml",
    ["models/main_models/bherer_los_stu_migration.yaml"])
admixture_model_bootstrap = pandas.read_csv("models/bootstrap_models/bherer_los_stu_admixture.csv")
migration_model_bootstrap = pandas.read_csv("models/bootstrap_models/bherer_los_stu_migration.csv")

groups = [
    [migration_model_mle, migration_model_bootstrap],
    [admixture_model_mle, admixture_model_bootstrap]
]

mapping = {
    "T_NDH": (r"$T_{\text{ND--H}}$ (ka)", 1e-3),
    "T_HN0": (r"$T_{\text{H} \to \text{Vin}}$ (ka) (fixed)", 1e-3), 
    "T_OOA": (r"$T_{\text{Yor--OOA}}$ (ka)", 1e-3),
    "T_LS": (r"$T_{\text{OOA--Stu}}$ (ka)", 1e-3),
    "T_NH": (r"$T_{\text{Vin} \to \text{OOA}}$ (ka) (fixed)", 1e-3), 
    "T_Ust": (r"$T_{\text{OOA--Ust}}$ (ka) (fixed)", 1e-3), 
    "N_Anc": (r"$N_{\text{A}}$", 1),
    "N_Vin": (r"$N_{\text{Vin}}$", 1),
    "N_MH": (r"$N_{\text{H}}$ ", 1),
    "N_Yor": (r"$N_{\text{Yor}}$", 1),
    "N_OOA": (r"$N_{\text{OOA}}$", 1),
    "N_Los": (r"$N_{\text{Los}}$", 1),
    "N_Ust": (r"$N_{\text{Ust}}$", 1),
    "N_Stu": (r"$N_{\text{Stu}}$", 1),
    "m_SY": (r"$m_{\text{Yor--Stu}}$", 1e5),
    "m_LY": (r"$m_{\text{Yor--Los}}$", 1e5),
    "p_H_N": (r"$f_{\text{H} \to \text{N}}$", 1),
    "p_N_OOA": (r"$f_{\text{Vin}\to\text{OOA}}$", 1),
    "m_LS": (r"$m_{\text{Los--Stu}} (\times 10^{-5})$", 1e5),
    "T_L_S": (r"$T_{\text{Los$\to$Stu}}$ (ka)", 1e-3),
    "p_L_S": (r"$f_{\text{Los}\to\text{Stu}}$", 1),
}

fixed_params = {
    "T_S": 2e6,
    "T_HN0": 2.5e5, 
    "T_NH": 48000, 
    "T_Ust": 47000, 
    "N_S": 20000, 
}

printout(mapping, fixed_params, groups)

$T_{\text{ND--H}}$ (ka) & 812 & 751 -- 862 & 820 & 761 -- 870 \\
$T_{\text{H} \to \text{Vin}}$ (ka) (fixed) & 250 & --- & 250 & --- \\
$T_{\text{Yor--OOA}}$ (ka) & 53.6 & 49.8 -- 56.7 & 53.8 & 51.1 -- 57.8 \\
$T_{\text{OOA--Stu}}$ (ka) & 52.3 & 48 -- 54 & 52.7 & 48.8 -- 56.1 \\
$T_{\text{Vin} \to \text{OOA}}$ (ka) (fixed) & 48 & --- & 48 & --- \\
$T_{\text{OOA--Ust}}$ (ka) (fixed) & 47 & --- & 47 & --- \\
$N_{\text{A}}$ & 16800 & 16100 -- 17800 & 16800 & 16100 -- 17800 \\
$N_{\text{Vin}}$ & 2350 & 2130 -- 2540 & 2310 & 2120 -- 2530 \\
$N_{\text{H}}$  & 29000 & 26600 -- 31800 & 29300 & 26900 -- 32400 \\
$N_{\text{Yor}}$ & 21300 & 11300 -- 49900 & 19700 & 11300 -- 38500 \\
$N_{\text{OOA}}$ & 94.7 & 40.1 -- 362 & 77.6 & 46.1 -- 353 \\
$N_{\text{Los}}$ & 957 & 817 -- 1210 & 1490 & 1270 -- 1730 \\
$N_{\text{Ust}}$ & 20000 & 519 -- 10000 & 2120 & 545 -- 20000 \\
$N_{\text{Stu}}$ & 4810 & 3350 -- 6470 & 5150 & 3020 -- 9720 \\
$m_{\text{Yor--Stu}}$ & 0.001 & 0.001 -- 6.46 & 0.0406 & 0.001 -- 5

In [8]:
"""
a comparison of the bherer and zhou full model MLE
"""


bherer_mle = dpluspy.inference.load_param_table("models/main_models/params_full.yaml",
    ["models/main_models/bherer_full.yaml"])
zhou_mle = dpluspy.inference.load_param_table("models/main_models/params_full.yaml",
    ["models/main_models/zhou_full.yaml"])
bherer_bootstrap = pandas.read_csv("models/bootstrap_models/bherer_full.csv")
zhou_bootstrap = pandas.read_csv("models/bootstrap_models/zhou_full.csv")

groups = [
    [bherer_mle, bherer_bootstrap],
    [zhou_mle, zhou_bootstrap],
]

mapping = {
    "T_S": (r"$T_{\text{NDH--S}}$ (ka) (fixed)", 1e-3),
    'T_NDH': (r'$T_{\text{ND--H}}$ (ka)', 1e-3),
    'T_ND': (r'$T_{\text{N--Den}}$ (ka)', 1e-3),
    "T_HN0": (r"$T_{\text{H} \to \text{N}}$ (ka) (fixed)", 1e-3), 
    'T_AWN': (r'$T_{\text{WN--Alt}}$ (ka)', 1e-3),
    "T_HN1": (r"$T_{\text{H} \to \text{WN}}$ (ka) (fixed)", 1e-3),
    'T_CV': (r'$T_{\text{Cha--Vin}}$ (ka) ', 1e-3),
    'T_OOA': (r'$T_{\text{Yor--OOA}}$ (ka)', 1e-3),
    'T_SL': (r'$T_{\text{OOA--Stu}}$ (ka)', 1e-3),
    "T_NH": (r"$T_{\text{Vin} \to \text{OOA}}$ (ka) (fixed)", 1e-3), 
    "T_Ust": (r"$T_{\text{OOA--Ust}}$ (ka) (fixed)", 1e-3), 
    'T_L_S': (r'$T_{\text{Los$\to$Stu}}$ (ka)', 1e-3),
    'N_Anc': (r'$N_{\text{A}}$', 1),
    "N_S": (r"$N_{\text{S}}$ (fixed)", 1),
    'N_ND': (r'$N_{\text{ND}}$', 1),
    'N_N': (r'$N_{\text{N}}$', 1),
    'N_MH': (r'$N_{\text{H}}$ ', 1),
    'N_OOA': (r'$N_{\text{OOA}}$', 1),
    'N_Alt': (r'$N_{\text{Alt}}$', 1),
    'N_Chag': (r'$N_{\text{Cha}}$', 1),
    'N_Vin': (r'$N_{\text{Vin}}$', 1),
    'N_Den': (r'$N_{\text{Den}}$', 1),
    'N_Los': (r'$N_{\text{Los}}$', 1),
    'N_Ust': (r'$N_{\text{Ust}}$', 1),
    'N_Stut': (r'$N_{\text{Stu}}$', 1),
    'N_Yor': (r'$N_{\text{Yor}}$', 1),
    'm_ND': (r'$m_{\text{N--Den}}$ ($\times 10^{-5}$)', 1e5),
    'm_LY': (r'$m_{\text{Yor--Los}}$ ($\times 10^{-5}$)', 1e5),
    'p_S_D': (r'$f_{\text{S} \to \text{D}}$', 1),
    'p_H_N': (r'$f_{\text{H} \to \text{N}}$', 1),
    'p_H_WN': (r'$f_{\text{H} \to \text{WN}}$', 1),
    'p_N_OOA': (r'$f_{\text{Vin}\to\text{OOA}}$', 1),
    'p_L_S': (r'$f_{\text{Los}\to\text{Stu}}$', 1)
}

fixed_params = {
    "T_S": 2e6,
    "T_HN0": 2.5e5, 
    "T_HN1": 1.1e5, 
    "T_NH": 48000, 
    "T_Ust": 47000, 
    "N_S": 20000, 
}

printout(mapping, fixed_params, groups)

$T_{\text{NDH--S}}$ (ka) (fixed) & 2000 & --- & 2000 & --- \\
$T_{\text{ND--H}}$ (ka) & 798 & 748 -- 827 & 748 & 721 -- 772 \\
$T_{\text{N--Den}}$ (ka) & 688 & 639 -- 734 & 718 & 684 -- 735 \\
$T_{\text{H} \to \text{N}}$ (ka) (fixed) & 250 & --- & 250 & --- \\
$T_{\text{WN--Alt}}$ (ka) & 123 & 117 -- 137 & 118 & 117 -- 125 \\
$T_{\text{H} \to \text{WN}}$ (ka) (fixed) & 110 & --- & 110 & --- \\
$T_{\text{Cha--Vin}}$ (ka)  & 60.5 & 57.3 -- 67.7 & 61.1 & 57.9 -- 64.4 \\
$T_{\text{Yor--OOA}}$ (ka) & 56.9 & 53.6 -- 60 & 59.3 & 53.4 -- 60.1 \\
$T_{\text{OOA--Stu}}$ (ka) & 54.7 & 49.7 -- 57.6 & 56.8 & 50.8 -- 58.2 \\
$T_{\text{Vin} \to \text{OOA}}$ (ka) (fixed) & 48 & --- & 48 & --- \\
$T_{\text{OOA--Ust}}$ (ka) (fixed) & 47 & --- & 47 & --- \\
$T_{\text{Los$\to$Stu}}$ (ka) & 29.4 & 13.2 -- 35.9 & 28.4 & 12.5 -- 35.3 \\
$N_{\text{A}}$ & 16500 & 15900 -- 17300 & 17200 & 16700 -- 17700 \\
$N_{\text{S}}$ (fixed) & 20000 & --- & 20000 & --- \\
$N_{\text{ND}}$ & 5060 & 3540 -- 5760 & 1940 & 1940 -

In [10]:
"""
a comparison of slow and fast mutation rate MLE for the full model
"""


slow_mle = dpluspy.inference.load_param_table("models/main_models/params_full.yaml",
    ["models/main_models/bherer_full_slow_mutation.yaml"])
fast_mle = dpluspy.inference.load_param_table("models/main_models/params_full.yaml",
    ["models/main_models/bherer_full_fast_mutation.yaml"])
slow_bootstrap = pandas.read_csv("models/bootstrap_models/bherer_full_slow_mutation.csv")
fast_bootstrap = pandas.read_csv("models/bootstrap_models/bherer_full_fast_mutation.csv")

groups = [
    [slow_mle, slow_bootstrap],
    [fast_mle, fast_bootstrap],
]

mapping = {
    "T_S": (r"$T_{\text{NDH--S}}$ (ka) (fixed)", 1e-3),
    'T_NDH': (r'$T_{\text{ND--H}}$ (ka)', 1e-3),
    'T_ND': (r'$T_{\text{N--Den}}$ (ka)', 1e-3),
    "T_HN0": (r"$T_{\text{H} \to \text{N}}$ (ka) (fixed)", 1e-3), 
    'T_AWN': (r'$T_{\text{WN--Alt}}$ (ka)', 1e-3),
    "T_HN1": (r"$T_{\text{H} \to \text{WN}}$ (ka) (fixed)", 1e-3),
    'T_CV': (r'$T_{\text{Cha--Vin}}$ (ka) ', 1e-3),
    'T_OOA': (r'$T_{\text{Yor--OOA}}$ (ka)', 1e-3),
    'T_SL': (r'$T_{\text{OOA--Stu}}$ (ka)', 1e-3),
    "T_NH": (r"$T_{\text{Vin} \to \text{OOA}}$ (ka) (fixed)", 1e-3), 
    "T_Ust": (r"$T_{\text{OOA--Ust}}$ (ka) (fixed)", 1e-3), 
    'T_L_S': (r'$T_{\text{Los$\to$Stu}}$ (ka)', 1e-3),
    'N_Anc': (r'$N_{\text{A}}$', 1),
    "N_S": (r"$N_{\text{S}}$ (fixed)", 1),
    'N_ND': (r'$N_{\text{ND}}$', 1),
    'N_N': (r'$N_{\text{N}}$', 1),
    'N_MH': (r'$N_{\text{H}}$ ', 1),
    'N_OOA': (r'$N_{\text{OOA}}$', 1),
    'N_Alt': (r'$N_{\text{Alt}}$', 1),
    'N_Chag': (r'$N_{\text{Cha}}$', 1),
    'N_Vin': (r'$N_{\text{Vin}}$', 1),
    'N_Den': (r'$N_{\text{Den}}$', 1),
    'N_Los': (r'$N_{\text{Los}}$', 1),
    'N_Ust': (r'$N_{\text{Ust}}$', 1),
    'N_Stut': (r'$N_{\text{Stu}}$', 1),
    'N_Yor': (r'$N_{\text{Yor}}$', 1),
    'm_ND': (r'$m_{\text{N--Den}}$ ($\times 10^{-5}$)', 1e5),
    'm_LY': (r'$m_{\text{Yor--Los}}$ ($\times 10^{-5}$)', 1e5),
    'p_S_D': (r'$f_{\text{S} \to \text{D}}$', 1),
    'p_H_N': (r'$f_{\text{H} \to \text{N}}$', 1),
    'p_H_WN': (r'$f_{\text{H} \to \text{WN}}$', 1),
    'p_N_OOA': (r'$f_{\text{Vin}\to\text{OOA}}$', 1),
    'p_L_S': (r'$f_{\text{Los}\to\text{Stu}}$', 1)
}

fixed_params = {
    "T_S": 2e6,
    "T_HN0": 2.5e5, 
    "T_HN1": 1.1e5, 
    "T_NH": 48000, 
    "T_Ust": 47000, 
    "N_S": 20000, 
}

printout(mapping, fixed_params, groups)

$T_{\text{NDH--S}}$ (ka) (fixed) & 2000 & --- & 2000 & --- \\
$T_{\text{ND--H}}$ (ka) & 921 & 879 -- 946 & 691 & 654 -- 731 \\
$T_{\text{N--Den}}$ (ka) & 856 & 819 -- 882 & 599 & 548 -- 650 \\
$T_{\text{H} \to \text{N}}$ (ka) (fixed) & 250 & --- & 250 & --- \\
$T_{\text{WN--Alt}}$ (ka) & 136 & 126 -- 152 & 116 & 115 -- 120 \\
$T_{\text{H} \to \text{WN}}$ (ka) (fixed) & 110 & --- & 110 & --- \\
$T_{\text{Cha--Vin}}$ (ka)  & 68.4 & 60.6 -- 95.9 & 59.9 & 57.3 -- 63.6 \\
$T_{\text{Yor--OOA}}$ (ka) & 57.7 & 53.1 -- 59.4 & 56.7 & 52.5 -- 58 \\
$T_{\text{OOA--Stu}}$ (ka) & 56.2 & 51.6 -- 58 & 52.8 & 48.6 -- 55.6 \\
$T_{\text{Vin} \to \text{OOA}}$ (ka) (fixed) & 48 & --- & 48 & --- \\
$T_{\text{OOA--Ust}}$ (ka) (fixed) & 47 & --- & 47 & --- \\
$T_{\text{Los$\to$Stu}}$ (ka) & 30.2 & 15 -- 36.4 & 28.6 & 10.1 -- 35.6 \\
$N_{\text{A}}$ & 19600 & 19100 -- 20400 & 14400 & 13700 -- 15000 \\
$N_{\text{S}}$ (fixed) & 20000 & --- & 20000 & --- \\
$N_{\text{ND}}$ & 3280 & 3030 -- 3580 & 5190 & 3690 -- 69

In [13]:
"""
a comparison between full models with/without superarchaic introgression
"""


with_sup_mle = dpluspy.inference.load_param_table("models/main_models/params_full.yaml",
    ["models/main_models/bherer_full.yaml"])
without_sup_mle = dpluspy.inference.load_param_table("models/main_models/params_full_without_sup_den.yaml",
    ["models/main_models/bherer_full_without_sup_den.yaml"])
with_sup_bootstrap = pandas.read_csv("models/bootstrap_models/bherer_full.csv")
without_sup_bootstrap = pandas.read_csv("models/bootstrap_models/bherer_full_without_sup_den.csv")

groups = [
    [with_sup_mle, without_sup_mle],
    [without_sup_mle, without_sup_bootstrap],
]

mapping = {
    "T_S": (r"$T_{\text{NDH--S}}$ (ka) (fixed)", 1e-3),
    'T_NDH': (r'$T_{\text{ND--H}}$ (ka)', 1e-3),
    'T_ND': (r'$T_{\text{N--Den}}$ (ka)', 1e-3),
    "T_HN0": (r"$T_{\text{H} \to \text{N}}$ (ka) (fixed)", 1e-3), 
    'T_AWN': (r'$T_{\text{WN--Alt}}$ (ka)', 1e-3),
    "T_HN1": (r"$T_{\text{H} \to \text{WN}}$ (ka) (fixed)", 1e-3),
    'T_CV': (r'$T_{\text{Cha--Vin}}$ (ka) ', 1e-3),
    'T_OOA': (r'$T_{\text{Yor--OOA}}$ (ka)', 1e-3),
    'T_SL': (r'$T_{\text{OOA--Stu}}$ (ka)', 1e-3),
    "T_NH": (r"$T_{\text{Vin} \to \text{OOA}}$ (ka) (fixed)", 1e-3), 
    "T_Ust": (r"$T_{\text{OOA--Ust}}$ (ka) (fixed)", 1e-3), 
    'T_L_S': (r'$T_{\text{Los$\to$Stu}}$ (ka)', 1e-3),
    'N_Anc': (r'$N_{\text{A}}$', 1),
    "N_S": (r"$N_{\text{S}}$ (fixed)", 1),
    'N_ND': (r'$N_{\text{ND}}$', 1),
    'N_N': (r'$N_{\text{N}}$', 1),
    'N_MH': (r'$N_{\text{H}}$ ', 1),
    'N_OOA': (r'$N_{\text{OOA}}$', 1),
    'N_Alt': (r'$N_{\text{Alt}}$', 1),
    'N_Chag': (r'$N_{\text{Cha}}$', 1),
    'N_Vin': (r'$N_{\text{Vin}}$', 1),
    'N_Den': (r'$N_{\text{Den}}$', 1),
    'N_Los': (r'$N_{\text{Los}}$', 1),
    'N_Ust': (r'$N_{\text{Ust}}$', 1),
    'N_Stut': (r'$N_{\text{Stu}}$', 1),
    'N_Yor': (r'$N_{\text{Yor}}$', 1),
    'm_ND': (r'$m_{\text{N--Den}}$ ($\times 10^{-5}$)', 1e5),
    'm_LY': (r'$m_{\text{Yor--Los}}$ ($\times 10^{-5}$)', 1e5),
    'p_H_N': (r'$f_{\text{H} \to \text{N}}$', 1),
    'p_H_WN': (r'$f_{\text{H} \to \text{WN}}$', 1),
    'p_N_OOA': (r'$f_{\text{Vin}\to\text{OOA}}$', 1),
    'p_L_S': (r'$f_{\text{Los}\to\text{Stu}}$', 1)
}

fixed_params = {
    "T_HN0": 2.5e5, 
    "T_HN1": 1.1e5, 
    "T_NH": 48000, 
    "T_Ust": 47000, 
}

printout(mapping, fixed_params, groups)

$T_{\text{NDH--S}}$ (ka) (fixed) &  &  &  &  \\
$T_{\text{ND--H}}$ (ka) & 798 & 792 -- 792 & 792 & 752 -- 847 \\
$T_{\text{N--Den}}$ (ka) & 688 & 678 -- 678 & 678 & 640 -- 751 \\
$T_{\text{H} \to \text{N}}$ (ka) (fixed) & 250 & --- & 250 & --- \\
$T_{\text{WN--Alt}}$ (ka) & 123 & 121 -- 121 & 121 & 116 -- 132 \\
$T_{\text{H} \to \text{WN}}$ (ka) (fixed) & 110 & --- & 110 & --- \\
$T_{\text{Cha--Vin}}$ (ka)  & 60.5 & 60.7 -- 60.7 & 60.7 & 56.7 -- 70.5 \\
$T_{\text{Yor--OOA}}$ (ka) & 56.9 & 58 -- 58 & 58 & 52.5 -- 60.6 \\
$T_{\text{OOA--Stu}}$ (ka) & 54.7 & 55.7 -- 55.7 & 55.7 & 49.6 -- 58.7 \\
$T_{\text{Vin} \to \text{OOA}}$ (ka) (fixed) & 48 & --- & 48 & --- \\
$T_{\text{OOA--Ust}}$ (ka) (fixed) & 47 & --- & 47 & --- \\
$T_{\text{Los$\to$Stu}}$ (ka) & 29.4 & 31 -- 31 & 31 & 11 -- 36 \\
$N_{\text{A}}$ & 16500 & 16900 -- 16900 & 16900 & 16000 -- 17500 \\
$N_{\text{S}}$ (fixed) &  &  &  &  \\
$N_{\text{ND}}$ & 5060 & 4240 -- 4240 & 4240 & 2750 -- 5360 \\
$N_{\text{N}}$ & 2800 & 2670 -- 26